# MODIS时序平滑与物候提取实验

统一流程：读取时序 → QA筛选 → 曲线平滑与重建 → 物候日期提取 → 当年制图 → 多年分析与产品比较。

三种平滑方法并列使用同一组QA后观测；先比较曲线，再应用日期定义。

## 0. 读取一条真实季节曲线

2024年固定位置116.30°E、40.60°N，不自动挑选像元，不宣称已验证地类。
原始指数按产品比例缩放；时间采用合成期标签的DOY，不是逐像元实际观测日。


In [ ]:
from pathlib import Path
import os, sys
import numpy as np
import pandas as pd
from scipy.special import expit
from scipy.optimize import least_squares, curve_fit
from scipy.signal import find_peaks
from IPython.display import display

%matplotlib inline

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT / "src"))
from vegetation_lab import modis_teaching_support as lab

YEAR = 2024
INDEX = "evi"
SAMPLE_LON, SAMPLE_LAT = 116.30, 40.60
YEARS = list(range(2001, 2026))
RUN_FULL_SPATIAL = False  # 全区25年需约406万次拟合；准备好长时计算后改为True。
SPATIAL_BACKEND = "numba"  # 编译多线程，无自动回退；手动改为"scipy"使用原多进程方法。
SPATIAL_N_JOBS = 12  # 空间块并行进程数；1为串行，先用2，内存充足可试4。
SPATIAL_BLOCK_ROWS = 1  # 每个任务处理的连续行数。
VERIFY_SPATIAL = os.environ.get("MODIS_VERIFY_SPATIAL") == "1"  # 仅自动测试使用，范围明确标注。
RUN_DIR = lab.new_run(ROOT)
catalog = lab.inventory(ROOT)
data = lab.load_year(catalog, YEAR)
point = data.sel(lon=SAMPLE_LON, lat=SAMPLE_LAT, method="nearest")
t = pd.DatetimeIndex(point.time.values).dayofyear.to_numpy(float)
t_daily = np.arange(1, pd.Timestamp(YEAR, 12, 31).dayofyear + 1, dtype=float)
raw = point[INDEX].values
display(pd.DataFrame({"日期": point.time.values, INDEX.upper(): raw,
                      "SummaryQA": point.summaryqa_raw.values, "VI_quality": point.vi_quality.values,
                      "Usefulness": point.vi_usefulness.values}))
print("实际格点：", float(point.lon), float(point.lat), "；运行目录：", RUN_DIR)
lab.show_curve(t, raw, raw, title=f"{YEAR} 固定像元原始{INDEX.upper()}", index=INDEX,
               destination=RUN_DIR / "01_raw.png")

## 1. QA筛选：保留可靠观测

比较原始与QA后曲线。筛除值保持NaN，时间坐标保留；首末缺测不通过插值外推。


In [ ]:
def teaching_mask(data, index, policy):
    raw = data[f"{index}_raw"]
    good = ((raw >= -2000) & (raw <= 10000)
            & (data.summaryqa_raw >= 0)
            & (data.summaryqa_raw <= policy["summary_max"])
            & (data.vi_quality <= policy["vi_quality_max"])
            & (data.vi_usefulness <= policy["usefulness_max"]))
    for flag in ("adjacent_cloud", "mixed_cloud", "possible_snow", "possible_shadow"):
        if policy[flag]:
            good = good & (data[flag] == 0)
    return good

In [ ]:
QA = {"summary_max": 1, "vi_quality_max": 1, "usefulness_max": 4,
      "adjacent_cloud": True, "mixed_cloud": True, "possible_snow": True, "possible_shadow": True}
good = teaching_mask(point, INDEX, QA)
y = point[INDEX].where(good).values
print("QA后有效观测：", np.isfinite(y).sum(), "/", len(y))
lab.show_curve(t, raw, y, title="同一像元：原始与QA后观测", index=INDEX,
               destination=RUN_DIR / "02_qa.png")

## 2. 平滑与曲线重建

### 2.1 Double Logistic函数拟合

本节只估计参数并重建曲线，第3节再用参数定义物候日期。

$$VI(t)=b+\frac{a}{1+e^{-k_1(t-t_1)}}-\frac{a}{1+e^{-k_2(t-t_2)}}.$$

| 代码参数 | 含义 |
|---|---|
| baseline=b | 背景 |
| amplitude=a | 分量幅度尺度，不等于整条曲线峰谷差 |
| rise=t1 | 上升分量中心，本实验定义为SOS |
| duration=t2-t1 | 两中心间隔；EOS=rise+duration |
| k_rise、k_fall | 上升和下降分量的速率参数 |

In [ ]:
def double_logistic(t, baseline, amplitude, rise, duration, k_rise, k_fall):
    t = np.asarray(t, dtype=float)
    return baseline + amplitude * (
        expit(k_rise * (t - rise)) - expit(k_fall * (t - rise - duration)))


def double_logistic_jac(t, baseline, amplitude, rise, duration, k_rise, k_fall):
    """Columns follow baseline, amplitude, rise, duration, k_rise, k_fall."""
    t = np.asarray(t, dtype=float)
    u, v = t - rise, t - rise - duration
    sr, sf = expit(k_rise * u), expit(k_fall * v)
    # expit(-z) avoids cancellation in 1 - expit(z) at saturation.
    dr = sr * expit(-k_rise * u)
    df = sf * expit(-k_fall * v)
    return np.column_stack((np.ones_like(t), sr - sf,
                            amplitude * (-k_rise * dr + k_fall * df),
                            amplitude * k_fall * df,
                            amplitude * u * dr, -amplitude * v * df))


In [ ]:
def fit_dl(t, y, analytic_jac=True):
    t, y = np.asarray(t, float), np.asarray(y, float)
    present = np.isfinite(t) & np.isfinite(y)
    t, y = t[present], y[present]
    if not len(y):
        return None  # No numerical input, not a phenology quality filter.
    initial = [float(np.min(y)), max(float(np.ptp(y)), 0.01), 100., 180., .05, .05]
    lower = [-1.5, 0., 1., 1., .001, .001]
    upper = [1.5, 3., 366., 366., 1., 1.]
    result = least_squares(
        lambda p: double_logistic(t, *p) - y,
        initial, bounds=(lower, upper), x_scale="jac", max_nfev=1000,
        jac=(lambda p: double_logistic_jac(t, *p)) if analytic_jac else "2-point")
    # Preserve the returned estimate even if the optimizer reached its limit.
    return result


In [ ]:
fit = fit_dl(t, y)
params = fit.x if fit is not None else np.full(6, np.nan)
fitted = double_logistic(t_daily, *params)
parameter_table = pd.DataFrame({"参数": ["b", "a", "t1", "duration", "k1", "k2"], "拟合值": params})
display(parameter_table)
print(fit.message if fit is not None else "没有可供拟合的观测")
baseline, amplitude, rise, duration, k_rise, k_fall = params
rise_component = baseline + amplitude * expit(k_rise * (t_daily-rise))
fall_component = amplitude * expit(k_fall * (t_daily-rise-duration))
lab.show_curve(t, raw, y, t_daily,
               {"背景+上升分量": rise_component, "被减去的下降分量": fall_component, "DL完整曲线": fitted},
               title="读取拟合参数，解释两分量如何形成完整季节", index=INDEX,
               destination=RUN_DIR / "03_dl_fit.png")

### 2.2 同一组观测：迭代SG、Whittaker与DL平滑比较

固定当前年份、像元、指数及 QA，比较三种曲线重建方法：

- **迭代 SG**：调用随包提供的迭代滤波实现。先在首末有效观测之间线性插值到每日，再执行趋势估计与迭代 SG；不是一次普通 SG。
- **Whittaker**：每日网格上真实观测权重为 1，缺测日为 0，通过二阶差分惩罚控制平滑程度。插值点不作为真实观测参与拟合。
- **DL**：沿用前面直接对 QA 后观测完成的拟合，不将 SG 或 Whittaker 结果再次送入 DL。

SG 的 51/31 点窗口在每日网格上对应 51/31 天，采用教学起始设置；Whittaker 的 λ=1000 也只是起始设置，均不是 MODIS 的最优参数。日插值不会增加原始观测的信息量，长时间缺测区间需谨慎解释。

为统一比较范围，三条曲线仅显示首末有效观测之间的部分，本节只观察重建曲线。第4.2节再在这一共同范围内采用相同的 **50%动态阈值**提取SOS/POS/EOS/LOS；共同范围与全年范围的基线可能不同，日期不能混为一组。


In [ ]:
from scipy.sparse import diags
from scipy.sparse.linalg import spsolve
from vegetation_lab import chen_sg_filter as chen
import hashlib, json

SG = dict(trend_window=51, trend_polyorder=2, sg_window=31, sg_polyorder=2, iterations=10)
WH_LAMBDA = 1000


def whittaker_daily(observed, lam):
    # 与实验03相同：观测日权重1，缺测日权重0，二阶差分惩罚。
    n = len(observed)
    weights = np.isfinite(observed).astype(float)
    if n < 3 or weights.sum() < 2:
        return np.full(n, np.nan)
    D = diags([np.ones(n-2), -2*np.ones(n-2), np.ones(n-2)],
              [0, 1, 2], shape=(n-2, n))
    W = diags(weights)
    curve = spsolve((W + lam * D.T @ D).tocsc(), weights * np.nan_to_num(observed))
    support = np.flatnonzero(weights)
    curve[:support[0]] = np.nan
    curve[support[-1]+1:] = np.nan
    return curve


In [ ]:
# 本节只计算当前单像元，不启动全区多年计算。
valid_smoothing = np.isfinite(t) & np.isfinite(y)
if valid_smoothing.sum() < 3:
    raise ValueError("QA后至少需要3个有效观测，请检查当前像元与QA设置。")

# MODIS合成期标签对应DOY；同日若有重复观测，取中位数。
smoothing_observations = pd.Series(y[valid_smoothing], index=t[valid_smoothing]).groupby(level=0).median()
smoothing_support = ((t_daily >= smoothing_observations.index.min())
                     & (t_daily <= smoothing_observations.index.max()))
smoothing_t = t_daily[smoothing_support]
smoothing_observed = smoothing_observations.reindex(smoothing_t).to_numpy(float)
smoothing_linear = np.interp(smoothing_t, smoothing_observations.index, smoothing_observations.values)
smoothing_curves = {
    "Chen SG": chen.chen_sg_filter(smoothing_linear, nan_policy="raise", restore_nan=False, **SG),
    "Whittaker": whittaker_daily(smoothing_observed, WH_LAMBDA),
    "DL": fitted[smoothing_support],
}
lab.show_curve(t, raw, y, smoothing_t, smoothing_curves,
               title=f"{YEAR} 同一像元：Chen SG、Whittaker与DL（共同观测范围）",
               index=INDEX, destination=RUN_DIR / "05b_smoothing_comparison.png")

print("共同DOY范围：", smoothing_t[0], "—", smoothing_t[-1],
      "；有效观测日数：", np.isfinite(smoothing_observed).sum())
print("SG参数：", SG, "；Whittaker λ：", WH_LAMBDA)

pd.DataFrame({"DOY": smoothing_t, "QA_observed": smoothing_observed,
              "linear_interpolation": smoothing_linear, **smoothing_curves}).to_csv(
    RUN_DIR / "smoothing_curves.csv", index=False, encoding="utf-8-sig")
_ = (RUN_DIR / "smoothing_configuration.json").write_text(json.dumps({
    "year": YEAR, "index": INDEX, "QA": QA,
    "sample": {"lon": float(point.lon), "lat": float(point.lat)},
    "SG": SG, "Whittaker_lambda": WH_LAMBDA,
    "Whittaker_weights": "1 at QA-valid observation days; 0 at missing days",
    "SG_input": "daily linear interpolation inside first/last QA-valid observation",
    "SG_script_sha256": hashlib.sha256(Path(chen.__file__).read_bytes()).hexdigest(),
    "DL_input": "original QA-valid observations; existing fit reused",
    "DL_parameters": params.tolist(),
    "support_DOY": [float(smoothing_t[0]), float(smoothing_t[-1])],
    "threshold_fraction": .5, "time": "DOY from composite labels"
}, ensure_ascii=False, indent=2), encoding="utf-8")


### 2.3 平滑参数对照练习

先比较峰形、春秋转折和稀疏观测区间，再每次只改变一项参数。

- SG：`trend_window=51`用于初始趋势估计，`sg_window=31`用于迭代平滑，单位均为每日网格上的天数。窗口保持奇数且大于多项式阶数。可将`SG["sg_window"]`改为15或61，再运行第2.2节比较单元。
- Whittaker：将`WH_LAMBDA`由1000改为100或10000，观察细小起伏与平滑程度的变化。
- 重新运行参数定义单元会恢复默认值。重跑会更新当前目录的图和表，实验对照前另存要保留的结果。

记录“修改参数、原曲线、新曲线、转折变化”。第4.2节用统一阈值检查这些曲线差异如何影响日期。曲线更光滑或更贴近观测不等于物候更准确。


## 3. 物候提取一：DL参数日期

三种平滑曲线已完成，现在定义物候事件。复用第2.1节同一次DL拟合：SOS=t1，EOS=t2，LOS=EOS−SOS。POS为每日拟合网格的最大值日期。日期单位DOY，LOS单位天。


In [ ]:
def dl_dates(params, t_daily):
    baseline, amplitude, rise, duration, k_rise, k_fall = params
    t1 = rise
    t2 = rise + duration
    fitted = double_logistic(t_daily, *params)
    SOS = t1
    EOS = t2
    POS = t_daily[np.argmax(fitted)]
    LOS = EOS - SOS
    return np.array([SOS, POS, EOS, LOS])

In [ ]:
def extract_dl(t, y, t_daily):
    fit = fit_dl(t, y)
    if fit is None:
        return np.full(4, np.nan)
    return dl_dates(fit.x, t_daily)

In [ ]:
main_dates = dl_dates(params, t_daily) if fit is not None else np.full(4, np.nan)
main_table = pd.DataFrame([main_dates], columns=lab.LABELS, index=["DL参数法"])
display(main_table)
lab.show_curve(t, raw, y, t_daily, {"DL拟合": fitted}, dict(zip(lab.LABELS[:3], main_dates[:3])),
               title="DL参数日期；DOY，LOS单位为天", index=INDEX, destination=RUN_DIR / "04_dates.png")
lab.save_tables(RUN_DIR, parameters=parameter_table, dl_dates=main_table.reset_index())

In [ ]:
lab.save_configuration(RUN_DIR, INDEX, QA, YEAR, point,
                       [teaching_mask, double_logistic, double_logistic_jac, fit_dl, dl_dates, extract_dl])

## 4. 物候提取二：动态阈值法

### 4.1 同一条DL曲线：参数日期与振幅阈值日期

峰前/峰后分别取曲线最低值作为基线，以各自振幅的指定比例求交叉。SOS取峰前最后一次向上穿越，EOS取峰后第一次向下穿越。

默认`THRESHOLD_FRACTION=0.5`。参考Word中的练习，将它改为0.3，仅重跑本节结果单元，观察日期与季长变化。标签会随参数更新。此处使用全年DL曲线，第4.2节仍固定共同观测范围和50%阈值。


In [ ]:
def threshold_dates(t, curve, fraction=.5):
    if not np.isfinite(curve).all():
        return np.full(4, np.nan), (np.nan, np.nan)
    peak = int(np.argmax(curve))
    base_left = np.min(curve[:peak + 1])
    base_right = np.min(curve[peak:])
    rise_level = base_left + fraction * (curve[peak] - base_left)
    fall_level = base_right + fraction * (curve[peak] - base_right)
    up = np.flatnonzero((curve[:-1] < rise_level) & (curve[1:] >= rise_level))
    down = np.flatnonzero((curve[:-1] >= fall_level) & (curve[1:] < fall_level))
    up, down = up[up < peak], down[down >= peak]

    def crossing(indices, level, first):
        if not len(indices):
            return np.nan
        i = indices[0] if first else indices[-1]
        return t[i] + (level - curve[i]) / (curve[i + 1] - curve[i]) * (t[i + 1] - t[i])

    SOS50 = crossing(up, rise_level, False)  # Last upward crossing before peak.
    EOS50 = crossing(down, fall_level, True)  # First downward crossing after peak.
    return np.array([SOS50, t[peak], EOS50, EOS50 - SOS50]), (rise_level, fall_level)

In [ ]:
THRESHOLD_FRACTION = 0.5  # 可改0.3，比较同一DL曲线的日期定义
threshold_result, levels = threshold_dates(t_daily, fitted, fraction=THRESHOLD_FRACTION)
comparison = pd.DataFrame([main_dates, threshold_result, threshold_result-main_dates],
                          index=["DL参数法", f"DL+{THRESHOLD_FRACTION:.0%}", "阈值减参数"], columns=lab.LABELS)
display(comparison)
marks = {"t1": main_dates[0], "t2": main_dates[2], f"SOS{THRESHOLD_FRACTION:.0%}": threshold_result[0], f"EOS{THRESHOLD_FRACTION:.0%}": threshold_result[2]}
lab.show_curve(t, raw, y, t_daily, {"同一次DL拟合": fitted}, marks, levels,
               title="同一曲线，两种事件定义", index=INDEX, destination=RUN_DIR / "05_threshold.png")
lab.save_tables(RUN_DIR, definition_comparison=comparison.reset_index())

### 4.2 三种平滑曲线采用相同50%阈值

复用第2.2节曲线，在共同观测范围内提取日期。先固定日期定义，再判断平滑方法造成的差异；缺少阈值交点保持NaN。


In [ ]:
smoothing_rows = []
for name, curve in smoothing_curves.items():
    dates, _ = threshold_dates(smoothing_t, curve, fraction=.5)
    smoothing_rows.append({"方法": name + "+50%", **dict(zip(lab.LABELS, dates))})
smoothing_dates = pd.DataFrame(smoothing_rows)
display(smoothing_dates.round(2))
smoothing_dates.to_csv(RUN_DIR / "smoothing_dates_50pct.csv", index=False, encoding="utf-8-sig")


## 5. 拓展二：分段Logistic几何特征法

### 5.1 局部斜率与增长/衰退分段

In [ ]:
def _local_slope(t, y, window=5):
    t=np.asarray(t,float);y=np.asarray(y,float);s=np.full(y.shape,np.nan)
    half=window//2
    for i in range(len(y)):
        lo=max(0,i-half);hi=min(len(y),i+half+1)
        ok=np.isfinite(t[lo:hi])&np.isfinite(y[lo:hi])
        if ok.sum()>=3:
            try:s[i]=np.polyfit(t[lo:hi][ok],y[lo:hi][ok],1)[0]
            except np.linalg.LinAlgError:pass
    return s

def detect_monotonic_segments(t,y,window=5,min_points=4):
    t=np.asarray(t,float);y=np.asarray(y,float);s=_local_slope(t,y,window)
    scale=np.nanpercentile(np.abs(s[np.isfinite(s)]),70) if np.isfinite(s).any() else 0
    eps=max(scale*0.05,1e-7)
    sign=np.where(s>eps,1,np.where(s<-eps,-1,0))
    segments=[];i=0
    while i<len(sign):
        if sign[i]==0:i+=1;continue
        direction=sign[i];j=i+1
        while j<len(sign) and sign[j]==direction:j+=1
        if j-i>=min_points:
            left=max(0,i-2);right=min(len(y)-1,j+1)
            ok=np.isfinite(t[left:right+1])&np.isfinite(y[left:right+1])
            if ok.sum()>=5:segments.append({"direction":"growth" if direction>0 else "senescence","start":left,"end":right})
        i=j
    # merge overlapping same-direction candidates
    merged=[]
    for seg in segments:
        if merged and seg['direction']==merged[-1]['direction'] and seg['start']<=merged[-1]['end']+1:
            merged[-1]['end']=max(merged[-1]['end'],seg['end'])
        else:merged.append(seg)
    return merged,s

In [ ]:
segments, local_slopes = detect_monotonic_segments(t, y, window=5, min_points=4)
display(pd.DataFrame(segments))

### 5.2 分段Logistic拟合、导数和极值

$$K=\frac{y''}{(1+y'^2)^{3/2}},\quad dK/ds=\frac{dK/dt}{\sqrt{1+y'^2}}.$$

增长与衰退分别拟合，显示各段候选极值。每段首尾极值为转折候选；
多个同方向段时沿用原实现的最大拟合幅度段选择。

In [ ]:
def logistic(t, a, b, c, d, t_ref=0.0):
    u = (np.asarray(t, dtype=float) - t_ref) / 365.0
    z = np.clip(a + b * u, -60, 60)
    return c / (1.0 + np.exp(z)) + d

In [ ]:
zhang_fits, zhang_rows = [], []
for segment in segments:
    selection = slice(segment["start"], segment["end"] + 1)
    ts, ys = t[selection], y[selection]
    valid = np.isfinite(ys)
    ts, ys = ts[valid], ys[valid]
    t_ref = float(np.mean(ts))
    growth = segment["direction"] == "growth"
    b0 = -8. if growth else 8.
    p0 = [-b0 * (np.median(ts)-t_ref)/365., b0, max(np.ptp(ys), .05), np.min(ys)]
    bounds = ([-40, -60 if growth else .0001, 0, -1.5],
              [40, -.0001 if growth else 60, 3, 1.5])
    try:
        popt, _ = curve_fit(lambda tt, a, b, c, d: logistic(tt, a, b, c, d, t_ref),
                            ts, ys, p0=p0, bounds=bounds, maxfev=30000)
    except (ValueError, RuntimeError) as error:
        print("该段无法拟合：", segment, str(error))
        continue
    dense = np.linspace(ts.min(), ts.max(), max(300, len(ts)*30))
    segment_curve = logistic(dense, *popt, t_ref=t_ref)
    d1 = np.gradient(segment_curve, dense, edge_order=2)
    d2 = np.gradient(d1, dense, edge_order=2)
    curvature = d2 / (1+d1*d1)**1.5
    curvature_rate = np.gradient(curvature, dense, edge_order=2) / np.sqrt(1+d1*d1)
    prominence = max(np.std(curvature_rate)*.05, 1e-12)
    high, _ = find_peaks(curvature_rate, prominence=prominence, distance=max(5, len(dense)//40))
    low, _ = find_peaks(-curvature_rate, prominence=prominence, distance=max(5, len(dense)//40))
    extrema = np.unique(np.r_[high, low])
    candidates = dense[extrema]
    onset, ending = (candidates[0], candidates[-1]) if len(candidates) >= 2 else (np.nan, np.nan)
    zhang_rows.append({"direction": segment["direction"], "amplitude": popt[2],
                       "onset": onset, "ending": ending, "candidates": candidates.tolist()})
    zhang_fits.append({"direction": segment["direction"], "dense": dense, "fitted": segment_curve,
                       "rate": curvature_rate, "extrema": extrema})
display(pd.DataFrame(zhang_rows))
zhang_dates = {}
for direction, names in [("growth", ["Greenup", "Maturity"]), ("senescence", ["Senescence", "Dormancy"])]:
    choices = [row for row in zhang_rows if row["direction"] == direction and np.isfinite(row["onset"])]
    chosen = max(choices, key=lambda row: row["amplitude"], default=None)
    zhang_dates.update(dict(zip(names, [chosen["onset"], chosen["ending"]] if chosen else [np.nan, np.nan])))
display(pd.DataFrame([zhang_dates]))
lab.show_zhang(t, y, local_slopes, zhang_fits, RUN_DIR / "06_zhang.png")

### 5.3 输入条件对照：一次只改指数或QA

保持DL方法与参数日期定义不变，只改`TRIAL_INDEX`（如`ndvi`）或一项`TRIAL_QA`。比较有效观测与日期，记录输入条件造成的差异。对应PPT第21页的实验记录任务。

In [ ]:
TRIAL_INDEX = INDEX  # 第二次可改为"ndvi"
TRIAL_QA = dict(QA)
# 第一次对照可设 TRIAL_QA["summary_max"] = 0
trial_y = point[TRIAL_INDEX].where(teaching_mask(point, TRIAL_INDEX, TRIAL_QA)).values
trial_dates = extract_dl(t, trial_y, t_daily)
display(pd.DataFrame([main_dates, trial_dates], columns=lab.LABELS, index=["主实验", "本次输入对照"]))

## 6. 从单像元到当年空间制图：SG 50%动态阈值

本节只计算 `YEAR` 指定的当年（默认2024），使用完整下载格网，与下一节的多年计算分开。输入仍为同一个指数 `INDEX` 和同一套 `QA`，SG参数沿用第2.2节的 `SG`。

单像元输入是一列“时间—植被指数”，空间输入则是 `时间 × 纬度 × 经度` 的数据立方体。**对每个格点重复“有效观测 → 支持范围内每日插值 → 迭代 SG → 50%动态阈值”即可得到一张日期地图。** 50%阈值分别使用峰值左、右两侧的最低值作为基线；SOS为峰前最后一次向上穿越，EOS为峰后第一次向下穿越，POS为平滑曲线最大值对应日。

每个像元至少需要3个QA有效观测，且只在自己的首末有效观测之间计算；缺测和无法找到的阈值交点保留NaN。本节不根据日期大小剔除异常像元，也不将空值填0。低振幅、多季或长缺测序列的物候含义需要结合原始曲线判断。

### 6.1 先把单像元步骤写成函数

下方函数与前面单像元对比采用同样的SG参数和阈值函数，只返回SOS、POS、EOS。LOS留给后面的独立单元显式计算。


In [ ]:
import xarray as xr
import matplotlib.pyplot as plt
from vegetation_lab.plot_style import configure_plot_fonts
import time


def sg50_pixel(observation_t, observation_y, daily, settings):
    valid = np.isfinite(observation_t) & np.isfinite(observation_y)
    if valid.sum() < 3:
        return np.full(3, np.nan)
    tx, yy = observation_t[valid], observation_y[valid]
    supported_t = daily[(daily >= tx[0]) & (daily <= tx[-1])]
    interpolated = np.interp(supported_t, tx, yy)
    smoothed = chen.chen_sg_filter(interpolated, nan_policy="raise", restore_nan=False, **settings)
    dates, _ = threshold_dates(supported_t, smoothed, fraction=.5)
    return dates[:3]


print("当前单像元SOS、POS、EOS：", sg50_pixel(t, y, t_daily, SG))


### 6.2 将同样的计算应用到每个格点

最直接的写法是两层循环：`result[:, row, col] = sg50_pixel(t, values[:, row, col], daily, SG)`。

下面为减少重复滤波开销，把首末有效观测位置相同的像元分组，每批最多256个。**各列仍独立插值、独立滤波、独立提取日期，不进行空间平均。** 最后将结果恢复为 `指标 × 纬度 × 经度`。单像元与批量结果将在当前样点自动核对。


In [ ]:
def sg50_grid(observation_t, values, daily, settings):
    # (时间, 纬度, 经度) → (时间, 像元)，只改变排列，不改变观测。
    flat = values.reshape(len(observation_t), -1)
    result = np.full((3, flat.shape[1]), np.nan)
    groups = {}
    for pixel in range(flat.shape[1]):
        valid = np.flatnonzero(np.isfinite(flat[:, pixel]))
        if len(valid) >= 3:
            groups.setdefault((valid[0], valid[-1]), []).append(pixel)
    completed = 0
    for (first, last), pixels in groups.items():
        supported_t = daily[(daily >= observation_t[first]) & (daily <= observation_t[last])]
        for offset in range(0, len(pixels), 256):
            batch = pixels[offset:offset+256]
            interpolated = np.empty((len(supported_t), len(batch)))
            for col, pixel in enumerate(batch):
                valid = np.isfinite(flat[:, pixel])
                interpolated[:, col] = np.interp(supported_t, observation_t[valid], flat[valid, pixel])
            smoothed = chen.chen_sg_filter(interpolated, nan_policy="raise", restore_nan=False, **settings)
            for col, pixel in enumerate(batch):
                dates, _ = threshold_dates(supported_t, smoothed[:, col], fraction=.5)
                result[:, pixel] = dates[:3]
            completed += len(batch)
        print(f"SG已处理 {completed}/{flat.shape[1]} 个像元（不足3个有效观测的像元保留空值）", end="\r")
    print()
    return result.reshape(3, *values.shape[1:])


def plot_current_maps(fields, titles, filename, units, limits, cmap="viridis"):
    configure_plot_fonts()
    fig, axes = plt.subplots(1, len(fields), figsize=(5*len(fields), 4.8),
                             layout="constrained", squeeze=False)
    colors = plt.get_cmap(cmap).copy()
    colors.set_bad("#eeeeee")
    for ax, field, title in zip(axes[0], fields, titles):
        # 使用坐标绘图，不假设纬度数组从北到南或从南到北。
        im = ax.pcolormesh(field.lon, field.lat, field.values, shading="auto",
                           cmap=colors, vmin=limits[0], vmax=limits[1], rasterized=True)
        ax.set(title=title, xlabel="经度（°E）", ylabel="纬度（°N）")
        fig.colorbar(im, ax=ax, shrink=.82, label=units, extend="both" if cmap=="RdBu_r" else "neither")
    fig.savefig(CURRENT_DIR / filename, dpi=180)
    plt.show()
    plt.close(fig)


In [ ]:
CURRENT_DIR = RUN_DIR / f"current_year_{YEAR}"
CURRENT_DIR.mkdir(exist_ok=True)
current_data = lab.load_year(catalog, YEAR)
current_values = current_data[INDEX].where(teaching_mask(current_data, INDEX, QA)).transpose("time", "lat", "lon").values
current_t = pd.DatetimeIndex(current_data.time.values).dayofyear.to_numpy(float)
current_daily = np.arange(1, pd.Timestamp(YEAR, 12, 31).dayofyear+1, dtype=float)
assert np.all(np.diff(current_t) > 0), "本节要求合成期日期按时间递增且不重复。"

started = time.perf_counter()
sg_dates_map = xr.DataArray(
    sg50_grid(current_t, current_values, current_daily, SG),
    dims=("metric", "lat", "lon"),
    coords={"metric": ["SOS", "POS", "EOS"], "lat": current_data.lat, "lon": current_data.lon})
sg_current = xr.Dataset({metric: sg_dates_map.sel(metric=metric, drop=True) for metric in ["SOS", "POS", "EOS"]})
sg_current.attrs.update(year=YEAR, index=INDEX, method="Chen SG + 50% dynamic threshold",
                        scope="full_downloaded_grid", time="DOY from composite labels",
                        support="first to last QA-valid observation per pixel")
for metric in sg_current:
    sg_current[metric].attrs["units"] = "DOY"
print(f"{YEAR} SG完成：{time.perf_counter()-started:.1f}秒；格网：{current_values.shape[1:]}")

row = int(np.argmin(np.abs(current_data.lat.values-SAMPLE_LAT)))
col = int(np.argmin(np.abs(current_data.lon.values-SAMPLE_LON)))
np.testing.assert_allclose(sg_dates_map.values[:, row, col],
    sg50_pixel(current_t, current_values[:, row, col], current_daily, SG), equal_nan=True, atol=1e-8)
print("当前样点的单像元结果与空间结果一致。")
sg_current.to_netcdf(CURRENT_DIR / "SG50_SOS_POS_EOS.nc", engine="h5netcdf")
plot_current_maps([sg_current[m] for m in ["SOS", "POS", "EOS"]],
                  [f"{YEAR} SG50% · {m}" for m in ["SOS", "POS", "EOS"]],
                  "01_SG50_SOS_POS_EOS.png", "年积日（DOY）", (1, len(current_daily)))


### 6.3 独立计算生长季长度：LOS = EOS − SOS

前一单元得到的是三个日期图层；现在对两张日期图逐像元相减，得到生长季长度，单位由DOY变为“天”。任何一端缺测，LOS也保持缺测。


In [ ]:
# 显式逐像元运算：xarray按相同经纬度格网对应相减。
sg_current["LOS"] = sg_current["EOS"] - sg_current["SOS"]
sg_current["LOS"].attrs.update(units="days", definition="EOS minus SOS")
sg_current.to_netcdf(CURRENT_DIR / "SG50_with_LOS.nc", engine="h5netcdf")
plot_current_maps([sg_current["LOS"]], [f"{YEAR} SG50% · LOS = EOS − SOS"],
                  "02_SG50_LOS.png", "生长季长度（天）", (0, len(current_daily)))


### 6.4 同年同格网：SG50% 与 DL参数法的空间差值

DL直接拟合同年、同一QA后的原始观测，沿用前面的 `extract_dl`（SOS=t1、EOS=t1+duration），不将SG结果作为DL输入。本单元独立计算当年DL，不依赖下一节的多年结果。后端沿用 `SPATIAL_BACKEND`；修改DL函数后请选择 `scipy`。

差值统一定义为 **SG50% − DL参数法**，单位为天。正值（红）表示SG日期更晚，负值（蓝）表示SG日期更早；两种方法均有有限日期值的像元才计算差值。这里同时包含曲线重建方法和日期定义的差异，是方法差异图，不能当作精度误差图。SG只使用观测支持范围，DL参数日期可能超出该范围，需结合曲线解释。

两张差值图使用同一对称色标，默认±60天；超出范围的数值只在显示时饱和，文件保留完整差值，表格列出超范围像元数。


In [ ]:
dl_current = lab.map_year(current_data, current_values, extract_dl,
                          n_jobs=SPATIAL_N_JOBS, block_rows=32, backend=SPATIAL_BACKEND)
dl_current.attrs.update(year=YEAR, index=INDEX, method="DL_parameter", backend=SPATIAL_BACKEND,
                        scope="full_downloaded_grid", definition="SOS=rise; EOS=rise+duration")
dl_current.to_dataset(name="phenology").to_netcdf(CURRENT_DIR / "DL_parameter.nc", engine="h5netcdf")

# exact要求两种结果坐标完全一致，避免无意中插值或对齐到不同格网。
sg_aligned, dl_aligned = xr.align(sg_current, dl_current, join="exact")
method_difference = xr.Dataset()
DIFF_LIMIT_DAYS = 60
comparison_rows = []
for metric in ["SOS", "EOS"]:
    sg_field = sg_aligned[metric]
    dl_field = dl_aligned.sel(metric=metric, drop=True)
    paired = np.isfinite(sg_field) & np.isfinite(dl_field)
    method_difference[metric] = (sg_field-dl_field).where(paired)
    method_difference[metric].attrs.update(units="days", definition="SG50 minus DL_parameter")
    comparison_rows.append({"指标": metric, "SG有效像元": int(np.isfinite(sg_field).sum()),
                            "DL有效像元": int(np.isfinite(dl_field).sum()), "配对像元": int(paired.sum()),
                            "超出色标范围像元": int((np.abs(method_difference[metric]) > DIFF_LIMIT_DAYS).sum())})
method_difference.attrs.update(year=YEAR, index=INDEX, difference="SG50 minus DL_parameter")
method_difference.to_netcdf(CURRENT_DIR / "SG50_minus_DL_dates.nc", engine="h5netcdf")
display(pd.DataFrame(comparison_rows))
pd.DataFrame(comparison_rows).to_csv(CURRENT_DIR / "comparison_coverage.csv", index=False, encoding="utf-8-sig")
plot_current_maps([method_difference[m] for m in ["SOS", "EOS"]],
                  [f"{YEAR} {m}差值：SG50% − DL参数法" for m in ["SOS", "EOS"]],
                  "03_SG50_minus_DL_SOS_EOS.png", "日期差（天）；正值=SG更晚",
                  (-DIFF_LIMIT_DAYS, DIFF_LIMIT_DAYS), cmap="RdBu_r")

_ = (CURRENT_DIR / "configuration.json").write_text(json.dumps({
    "year": YEAR, "index": INDEX, "QA": QA, "SG": SG, "threshold_fraction": .5,
    "scope": "full_downloaded_grid", "shape_lat_lon": list(current_values.shape[1:]),
    "SG_min_valid_observations": 3, "SG_support": "per-pixel first/last valid observation; no extrapolation",
    "SG_script_sha256": hashlib.sha256(Path(chen.__file__).read_bytes()).hexdigest(),
    "DL_backend": SPATIAL_BACKEND, "DL_n_jobs": SPATIAL_N_JOBS, "DL_block_rows": 32,
    "difference": "SG50 minus DL_parameter", "difference_display_limit_days": DIFF_LIMIT_DAYS,
    "missing": "NaN; difference requires finite dates from both methods"
}, ensure_ascii=False, indent=2), encoding="utf-8")
print("当年空间结果：", CURRENT_DIR)


### 6.5 同年同格网：SG与DL散点对比及统计指标

复用第6.4节已经对齐的当年地图，不重复拟合。分别对SOS和EOS选取两种方法均为有限日期值的像元，横轴为DL参数日期，纵轴为SG50%日期。图中包含全部有效配对，1∶1线表示两种方法日期相同；不根据日期范围或差值大小删点。

令$x_i$为DL日期，$y_i$为SG日期，$d_i=y_i-x_i$。采用与第9节一致的指标定义：

$$\mathrm{Bias}=\frac{1}{n}\sum_i d_i,\quad
\mathrm{MAE}=\frac{1}{n}\sum_i|d_i|,\quad
\mathrm{RMSE}=\sqrt{\frac{1}{n}\sum_i d_i^2},\quad
R^2=1-\frac{\sum_i d_i^2}{\sum_i(x_i-\bar{x})^2}.$$

日期单位为DOY，Bias、MAE和RMSE单位为天。正Bias表示SG日期整体偏晚。R²以DL作为比较基准，可能为负，**不是相关系数的平方**；DL日期没有方差或有效配对不足时R²为NaN。

这些指标量化的是SG50%与DL参数法的差异，包含平滑与日期定义两方面的影响，不能把DL当作地面真值来解释。散点图对应修订版PPT第20页。


In [ ]:
configure_plot_fonts()
sg_dl_rows = []
fig, axes = plt.subplots(1, 2, figsize=(12, 5.2), layout="constrained")
for ax, metric in zip(axes, ["SOS", "EOS"]):
    dl_values = dl_aligned.sel(metric=metric, drop=True).values.ravel()
    sg_values = sg_aligned[metric].values.ravel()
    paired = np.isfinite(dl_values) & np.isfinite(sg_values)
    x, y_sg = dl_values[paired], sg_values[paired]
    difference = y_sg - x
    n_pairs = len(x)
    sst = np.sum((x - x.mean()) ** 2) if n_pairs else 0.0
    bias = difference.mean() if n_pairs else np.nan
    mae = np.abs(difference).mean() if n_pairs else np.nan
    rmse = np.sqrt(np.mean(difference ** 2)) if n_pairs else np.nan
    r2 = 1 - np.sum(difference ** 2) / sst if n_pairs > 1 and sst > 0 else np.nan
    sg_dl_rows.append({"metric": metric, "n": n_pairs, "Bias_days": bias,
                       "MAE_days": mae, "RMSE_days": rmse, "R2_DL_reference": r2})

    ax.scatter(x, y_sg, s=3, alpha=.18, color="#2475A8", linewidths=0,
               rasterized=True)
    if n_pairs:
        lower, upper = min(x.min(), y_sg.min()), max(x.max(), y_sg.max())
        margin = max((upper - lower) * .04, 1.0)
        limits = (lower - margin, upper + margin)
        ax.plot(limits, limits, "--", color=".4", linewidth=1.2, label="1∶1线")
        ax.set_xlim(limits)
        ax.set_ylim(limits)
        ax.legend(loc="lower right", frameon=False)
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlabel("DL参数日期（DOY）")
    ax.set_ylabel("SG50%日期（DOY）")
    ax.set_title(f"{metric}   n={n_pairs:,}", fontsize=13)
    ax.text(.03, .97, f"RMSE={rmse:.2f}天   MAE={mae:.2f}天\n"
            f"R²={r2:.3f}   Bias={bias:.2f}天", transform=ax.transAxes,
            va="top", fontsize=10,
            bbox=dict(facecolor="white", alpha=.9, edgecolor="none", pad=3))
    ax.grid(alpha=.15)

fig.suptitle(f"{YEAR} 同年同格网：SG50%与DL参数法（全部有效配对）", fontsize=14)
fig.savefig(CURRENT_DIR / "04_SG50_vs_DL_scatter.png", dpi=180)
plt.show()
plt.close(fig)
sg_dl_statistics = pd.DataFrame(sg_dl_rows)
display(sg_dl_statistics.round(3))
sg_dl_statistics.to_csv(CURRENT_DIR / "SG50_vs_DL_metrics.csv", index=False, encoding="utf-8-sig")
_ = (CURRENT_DIR / "SG50_vs_DL_metrics_definition.json").write_text(json.dumps({
    "year": YEAR, "index": INDEX, "x": "DL_parameter", "y": "SG50",
    "metrics": ["SOS", "EOS"], "pairing": "same grid; finite dates in both methods; no date/difference filtering",
    "scope": "full_downloaded_grid", "difference": "SG50 minus DL_parameter",
    "R2": "1 - sum((SG-DL)**2) / sum((DL-mean(DL))**2); not squared correlation",
    "units": {"dates": "DOY", "Bias_MAE_RMSE": "days", "R2": "dimensionless"},
    "interpretation": "method agreement; DL is not ground truth"
}, ensure_ascii=False, indent=2), encoding="utf-8")


## 7. 同一DL算法的多年与空间应用

### 7.1 固定像元2001—2025年

In [ ]:
annual_rows = []
for year in YEARS:
    annual = lab.load_year(catalog, year)
    sample = annual.sel(lon=SAMPLE_LON, lat=SAMPLE_LAT, method="nearest")
    obs_t = pd.DatetimeIndex(sample.time.values).dayofyear.to_numpy(float)
    daily_t = np.arange(1, pd.Timestamp(year, 12, 31).dayofyear+1, dtype=float)
    obs_y = sample[INDEX].where(teaching_mask(sample, INDEX, QA)).values
    row = dict(zip(lab.LABELS, extract_dl(obs_t, obs_y, daily_t)))
    annual_rows.append({"year": year, **row})
annual_table = pd.DataFrame(annual_rows)
display(annual_table)
lab.show_annual(annual_table, RUN_DIR / "07_annual.png")
lab.save_tables(RUN_DIR, annual_point=annual_table)

### 7.2 完整下载范围逐像元应用

In [ ]:
if RUN_FULL_SPATIAL or VERIFY_SPATIAL:
    spatial_cube = lab.spatial_years(ROOT, catalog, YEARS, INDEX, QA, teaching_mask, extract_dl,
                                     RUN_DIR, verify=VERIFY_SPATIAL,
                                     n_jobs=SPATIAL_N_JOBS, block_rows=SPATIAL_BLOCK_ROWS,
                                     backend=SPATIAL_BACKEND)
    for year in spatial_cube.year.values:
        if year in [2001, 2013, 2025]:
            lab.show_maps(spatial_cube.sel(year=year), f"{year}", RUN_DIR / f"map_{year}.png")
else:
    spatial_cube = None
    print("全区25年DL尚未运行；无全区地图或趋势结果。正式入口已保留。")

### 7.3 多年均值、年际变化与线性趋势

下面固定像元趋势可直接计算；全区统计只在逐年DL实际完成后产生。
不设置有效年份门槛，也不强制输出有效年份数。缺失不填0，无法计算的趋势保持空值。
SOS/POS/EOS趋势单位为天/年，负值表示日期提前；LOS负值表示季节缩短，不称为物候提前。
数值异常留给下一节思考，不先按生态合理性筛除。

In [ ]:
summary_rows = []
for metric in lab.LABELS:
    values = annual_table[metric].to_numpy(float)
    years = annual_table.year.to_numpy(float)
    present = np.isfinite(values)
    xv, yv = years[present], values[present]
    slope = np.polyfit(xv, yv, 1)[0] if len(np.unique(xv)) > 1 else np.nan
    summary_rows.append({"指标": metric, "均值": yv.mean() if len(yv) else np.nan,
                         "标准差": yv.std() if len(yv) else np.nan, "趋势(天/年)": slope})
display(pd.DataFrame(summary_rows))
if spatial_cube is not None:
    spatial_summary = lab.summarize_years(spatial_cube)
    spatial_summary.to_netcdf(RUN_DIR / "DL_multiyear_summary.nc", engine="h5netcdf")
    for statistic in ["mean", "std", "slope"]:
        lab.show_maps(spatial_summary[statistic], f"{statistic}", RUN_DIR / f"{statistic}.png")
lab.save_tables(RUN_DIR, annual_summary=pd.DataFrame(summary_rows))

## 8. 方法适用性思考

**Double Logistic是否完全适合北京市的物候提取？为什么？**

空间计算后选择不同形态或异常结果的像元；也可先用下方位置回看真实观测和拟合。

1. 哪些曲线不符合单季DL假设？请提供原始/拟合曲线证据。
2. 异常日期或趋势来自地表变化、观测质量还是模型适用性？
3. 若用于正式科研，你会如何改进，为什么？

In [ ]:
INSPECT_LON, INSPECT_LAT = SAMPLE_LON, SAMPLE_LAT  # 学生依据地图改位置，不自动选点
inspect_point = data.sel(lon=INSPECT_LON, lat=INSPECT_LAT, method="nearest")
inspect_y = inspect_point[INDEX].where(teaching_mask(inspect_point, INDEX, QA)).values
inspect_fit = fit_dl(t, inspect_y)
inspect_curve = double_logistic(t_daily, *inspect_fit.x) if inspect_fit is not None else np.full_like(t_daily, np.nan)
print("实际检查格点：", float(inspect_point.lon), float(inspect_point.lat))
lab.show_curve(t, inspect_point[INDEX].values, inspect_y, t_daily, {"DL拟合": inspect_curve},
               title="自主观察：该位置是否适合单季DL？", index=INDEX, destination=RUN_DIR / "08_inspect.png")

## 9. 拓展三：自己提取的物候与MCD12Q2是否一致？

这一节使用已下载的MCD12Q2.061参考产品，补齐空间/年份匹配、QA筛选、日期差值与误差评价。它是产品一致性比较，不是地面真值验证。

| 本实验 | 参考产品第一周期 | 比较含义 |
|---|---|---|
| DL＋50%的SOS/EOS | MidGreenup_1 / MidGreendown_1 | 上升/下降段50%振幅日期 |
| POS | Peak_1 | 同一生长周期的峰值日期 |
| 50%定义的LOS | MidGreendown_1 − MidGreenup_1 | 两个50%日期之间的间隔 |
| DL参数t1/t2 | 同上，作为辅助对照 | 参数中心与产品阈值日期的差异，不声称定义完全一致 |

产品基于NBAR-EVI2，本实验使用MOD13A1 EVI；即使同为50%，拟合、振幅基准和观测处理仍有差异。

课堂默认范围：固定像元2001—2024年；2024年在349×466原格网上每10行、每10列取一个像元。

依据：[GEE产品目录](https://developers.google.com/earth-engine/datasets/catalog/MODIS_061_MCD12Q2)。

In [ ]:
from vegetation_lab import phenology_product as product

PRODUCT_DIR = ROOT / "data/beijing_mcd12q2"
PRODUCT_OUTPUT = RUN_DIR / "product_comparison"
PRODUCT_OUTPUT.mkdir()
COMPARISON_YEAR = 2024
SAMPLE_STEP = 10  # 1为全格网；10为固定规则抽样，不依据误差选点
PRODUCT_QA_MAX = 1  # 0=Best，1=Good；改0可观察严格QA的样本覆盖变化
assert isinstance(SAMPLE_STEP, int) and SAMPLE_STEP >= 1
reference_manifest = pd.read_csv(PRODUCT_DIR / "MCD12Q2_band_manifest.csv")
comparison_years = sorted(set(reference_manifest.year) & set(catalog.year))
display(reference_manifest[reference_manifest.year.eq(COMPARISON_YEAR)][
    ["band_index_1based", "band_name", "unit_after_scaling", "scale_factor_to_apply"]])
print("共同年份：", comparison_years)
print("未配对的MOD13A1年份：", sorted(set(catalog.year) - set(comparison_years)))

### 9.1 转换日期，并明示哪些像元可以配对

原始日期是自1970-01-01起的天数。减去产品年1月1日距该起点的天数，再加1，得到相对该年的日序；闰年按实际天数计算，不取模。

本单季、年内实验选择NumCycles=1，整体QA及SOS/POS/EOS各自QA均≤1；日期须在本年内且SOS<POS<EOS。

In [ ]:
def reference_dates(bands, year, qa_max=1):
    """First-cycle 50% dates, with explicit single-season and QA eligibility."""
    epoch_days = (pd.Timestamp(year, 1, 1) - pd.Timestamp(1970, 1, 1)).days
    year_days = pd.Timestamp(year, 12, 31).dayofyear
    single = bands['NumCycles'] == 1
    overall = bands['QA_Overall_1']
    detailed = bands['QA_Detailed_1']
    qa_present = np.isfinite(detailed) & (detailed >= 0) & (detailed <= 16383)
    qa_integer = np.where(qa_present, detailed, 0).astype(np.int32)
    quality = single & (overall >= 0) & (overall <= qa_max) & qa_present
    dates = {}
    for metric, band, shift in [('SOS', 'MidGreenup_1', 2), ('POS', 'Peak_1', 6),
                               ('EOS', 'MidGreendown_1', 10)]:
        raw = bands[band]
        valid = np.isfinite(raw) & (raw >= 11138) & (raw <= 32766)
        dates[metric] = np.where(valid, raw - epoch_days + 1, np.nan)
        quality &= ((qa_integer >> shift) & 3) <= qa_max
    in_year = quality.copy()
    for values in dates.values():
        in_year &= np.isfinite(values) & (values >= 1) & (values <= year_days)
    eligible = in_year & (dates['SOS'] < dates['POS']) & (dates['POS'] < dates['EOS'])
    dates['LOS'] = dates['EOS'] - dates['SOS']
    counts = {'all_pixels': single.size, 'single_cycle': int(single.sum()),
              'single_good_QA': int(quality.sum()), 'in_year': int(in_year.sum()),
              'ordered_reference': int(eligible.sum())}
    return dates, eligible, counts

In [ ]:
reference_bands, grid_audit = product.read_reference(
    PRODUCT_DIR, COMPARISON_YEAR, catalog.loc[catalog.year.eq(COMPARISON_YEAR), "tif"].iloc[0])
ref_dates, ref_eligible, ref_counts = reference_dates(reference_bands, COMPARISON_YEAR, PRODUCT_QA_MAX)
display(pd.DataFrame([grid_audit]))
display(pd.DataFrame([ref_counts]))

### 9.2 同一原格点、同一年：复用已有DL拟合与50%阈值

从左上角开始固定规则抽样，再取通过参考产品筛选的点。

In [ ]:
def own_comparison_dates(observation_t, observation_y, daily_t):
    result = fit_dl(observation_t, observation_y)
    if result is None:
        return np.full(4, np.nan), np.full(4, np.nan), False
    parameter_dates = dl_dates(result.x, daily_t)
    curve = double_logistic(daily_t, *result.x)
    half_dates, _ = threshold_dates(daily_t, curve, fraction=.5)
    return parameter_dates, half_dates, bool(result.success)

comparison_data = lab.load_year(catalog, COMPARISON_YEAR)
comparison_t = pd.DatetimeIndex(comparison_data.time.values).dayofyear.to_numpy(float)
comparison_daily = np.arange(1, pd.Timestamp(COMPARISON_YEAR, 12, 31).dayofyear + 1, dtype=float)
comparison_values = comparison_data[INDEX].where(teaching_mask(comparison_data, INDEX, QA)).values
sample_mask = np.zeros(ref_eligible.shape, dtype=bool)
sample_mask[::SAMPLE_STEP, ::SAMPLE_STEP] = True
sample_rows = []
for rr, cc in np.argwhere(sample_mask & ref_eligible):
    obs = comparison_values[:, rr, cc]
    parameter_dates, half_dates, success = own_comparison_dates(comparison_t, obs, comparison_daily)
    record = {"year": COMPARISON_YEAR, "row": int(rr), "col": int(cc),
              "lon": float(comparison_data.lon[cc]), "lat": float(comparison_data.lat[rr]),
              "n_observations": int(np.isfinite(obs).sum()), "optimizer_success": success}
    for k, metric in enumerate(lab.LABELS):
        record.update({"ref_" + metric: ref_dates[metric][rr, cc],
                       "parameter_" + metric: parameter_dates[k], "threshold_" + metric: half_dates[k]})
    sample_rows.append(record)
spatial_pairs = pd.DataFrame(sample_rows)
if spatial_pairs.empty:
    raise ValueError("当前规则抽样没有通过参考QA的单季像元，请检查覆盖表与产品")
coverage = dict(ref_counts, sampled_pixels=int(sample_mask.sum()),
                sampled_reference_eligible=len(spatial_pairs),
                fitted_pixels=int(spatial_pairs.n_observations.gt(0).sum()),
                optimizer_success=int(spatial_pairs.optimizer_success.sum()))
display(pd.DataFrame([coverage]))
spatial_pairs.to_csv(PRODUCT_OUTPUT / "spatial_pairs.csv", index=False)
pd.DataFrame([coverage]).to_csv(PRODUCT_OUTPUT / "spatial_coverage.csv", index=False)

### 9.3 固定像元2001—2024年对比

In [ ]:
point_rows, grid_rows = [], []
for comparison_y in comparison_years:
    yearly = lab.load_year(catalog, comparison_y)
    rr = int(np.argmin(np.abs(yearly.lat.values - SAMPLE_LAT)))
    cc = int(np.argmin(np.abs(yearly.lon.values - SAMPLE_LON)))
    bands, audit = product.read_reference(PRODUCT_DIR, comparison_y,
        catalog.loc[catalog.year.eq(comparison_y), "tif"].iloc[0])
    dates, eligible, counts = reference_dates(bands, comparison_y, PRODUCT_QA_MAX)
    grid_rows.append(audit)
    selected_point = yearly.isel(lat=rr, lon=cc)
    obs_t = pd.DatetimeIndex(selected_point.time.values).dayofyear.to_numpy(float)
    daily = np.arange(1, pd.Timestamp(comparison_y, 12, 31).dayofyear + 1, dtype=float)
    obs = selected_point[INDEX].where(teaching_mask(selected_point, INDEX, QA)).values
    parameter_dates, half_dates, success = own_comparison_dates(obs_t, obs, daily)
    record = {"year": comparison_y, "row": rr, "col": cc,
              "lon": float(selected_point.lon), "lat": float(selected_point.lat),
              "reference_eligible": bool(eligible[rr, cc]), "optimizer_success": success}
    for k, metric in enumerate(lab.LABELS):
        record.update({"ref_" + metric: dates[metric][rr, cc],
                       "parameter_" + metric: parameter_dates[k], "threshold_" + metric: half_dates[k]})
    point_rows.append(record)
annual_pairs = pd.DataFrame(point_rows)
display(annual_pairs[["year", "reference_eligible", "ref_SOS", "threshold_SOS", "ref_EOS", "threshold_EOS"]])
annual_pairs.to_csv(PRODUCT_OUTPUT / "fixed_pixel_pairs.csv", index=False)
pd.DataFrame(grid_rows).to_csv(PRODUCT_OUTPUT / "grid_audit.csv", index=False)

### 9.4 指标、散点图与抽样差值空间图

差值=自行提取−参考产品，正Bias表示日期偏晚；对LOS则表示间隔偏长。MAE与RMSE单位均为天。
这里R²=1−Σ(自行提取−参考)²/Σ(参考−参考均值)²，可能为负，不是相关系数的平方。n是实际有效配对数。

In [ ]:
def comparison_scores(reference, estimate):
    reference, estimate = np.asarray(reference, float), np.asarray(estimate, float)
    valid = np.isfinite(reference) & np.isfinite(estimate)
    ref, est = reference[valid], estimate[valid]
    error = est - ref
    sst = np.sum((ref - ref.mean()) ** 2) if len(ref) else 0
    return {'n': len(ref), 'Bias': error.mean() if len(ref) else np.nan,
            'MAE': np.abs(error).mean() if len(ref) else np.nan,
            'RMSE': np.sqrt(np.mean(error ** 2)) if len(ref) else np.nan,
            'R2': 1 - np.sum(error ** 2) / sst if len(ref) > 1 and sst > 0 else np.nan}

In [ ]:
score_rows = []
for scope, table in [("spatial_sample", spatial_pairs),
                     ("fixed_pixel_years", annual_pairs[annual_pairs.reference_eligible])]:
    for metric in lab.LABELS:
        valid = np.isfinite(table[["ref_" + metric, "threshold_" + metric,
                                   "parameter_" + metric]]).all(axis=1)
        for method in ["threshold", "parameter"]:
            score_rows.append({"scope": scope, "method": method, "metric": metric,
                **comparison_scores(table.loc[valid, "ref_" + metric], table.loc[valid, method + "_" + metric])})
scores = pd.DataFrame(score_rows)
display(scores.round(3))
scores.to_csv(PRODUCT_OUTPUT / "comparison_metrics.csv", index=False)
product.plot_comparison(spatial_pairs, annual_pairs, PRODUCT_OUTPUT, COMPARISON_YEAR, SAMPLE_STEP)
import json
(PRODUCT_OUTPUT / "configuration.json").write_text(json.dumps({
    "product": "MODIS/061/MCD12Q2", "index": INDEX, "input_QA": QA,
    "reference_QA_max": PRODUCT_QA_MAX, "single_cycle_only": True,
    "reference_dates": ["MidGreenup_1", "Peak_1", "MidGreendown_1"],
    "spatial_year": COMPARISON_YEAR, "sample_step": SAMPLE_STEP,
    "spatial_scope": "regular_original_grid_pixel_sample" if SAMPLE_STEP > 1 else "full_grid",
    "fixed_pixel_years": comparison_years, "difference": "own_minus_reference",
    "reference_filter": "single cycle; overall and date QA; ordered within-year dates",
    "interpretation": "product consistency, not independent ground validation"
}, ensure_ascii=False, indent=2), encoding="utf8")
print("产品对比结果：", PRODUCT_OUTPUT)

### 9.5 如何解释差异？

先看n及覆盖表，再判断日期差异。50%定义相近也不保证一致；EVI与EVI2、拟合方式、合成期时间标签、雪云处理及产品相邻年份补缺都可能产生差异，不能仅由散点图判定原因。

讨论：参数法与50%法哪一种更接近产品？SOS和EOS的偏差是否同向？严格QA是否改变可比较样本的空间分布？图上空白是没有植被，还是没有进入当前对比？

进一步的独立验证可使用地面物候站、PhenoCam或通量塔，但需先统一事件定义、观测尺度及像元代表性。

## 参考文献

1. Zhang, X., Friedl, M. A., Schaaf, C. B., Strahler, A. H., Hodges, J. C. F., Gao, F., Reed, B. C., & Huete, A. (2003). Monitoring vegetation phenology using MODIS. *Remote Sensing of Environment*, **84**(3), 471–475. [DOI: 10.1016/S0034-4257(02)00135-9](https://doi.org/10.1016/S0034-4257(02)00135-9).
2. Chen, J., Jönsson, P., Tamura, M., Gu, Z., Matsushita, B., & Eklundh, L. (2004). A simple method for reconstructing a high-quality NDVI time-series data set based on the Savitzky–Golay filter. *Remote Sensing of Environment*, **91**, 332–344. [DOI: 10.1016/j.rse.2004.03.014](https://doi.org/10.1016/j.rse.2004.03.014).

文献1对应分段Logistic与几何特征提取，文献2对应迭代SG平滑。课堂数据、参数及数值实现以本实验代码和方法说明为准。
